In [1]:
"""
结果汇总脚本 v2
===============
读取四个模型的结果并生成对比表：
  - XGBoost  (results_xgb_{target}/)
  - LightGBM (results_lgb_{target}/)
  - Logistic (results_lr_{target}/)
  - CNN-GRU  (results_cnn_gru_{target}/)

输出:
  results_summary/model_comparison_{target}.csv   ← 四模型对比
  results_summary/best_model_shap_{target}.csv    ← 最优模型SHAP
  results_summary/overall_summary.csv             ← 全局汇总
"""

import pandas as pd
import os

os.makedirs('results_summary', exist_ok=True)

TARGETS = ['EWS_5min', 'EWS_15min', 'EWS_30min']

MODELS = {
    'XGBoost':  'results_xgb',
    'LightGBM': 'results_lgb',
    'Logistic': 'results_lr',
    'CNN-GRU':  'results_cnn_gru',
}

EXPERIMENT_FEATURES = {
    'baseline_market':        'market',
    'M1_add_cascade':         'market + cascade',
    'M2_add_network':         'market + cascade + network',
    'M3_add_temporal':        'market + cascade + network + temporal',
    'M4_add_text':            'market + cascade + network + temporal + text',
    'full_with_coordination': 'market + cascade + network + temporal + text + coordination',
}

overall_rows = []

for target in TARGETS:
    print(f"\n{'='*70}")
    print(f"  {target}")
    print(f"{'='*70}")

    # ── Table 1: 四模型 ablation 对比 ────────────────────────────────────────
    model_results = {}
    for model_name, prefix in MODELS.items():
        path = f'{prefix}_{target}/ablation_results.csv'
        if os.path.exists(path):
            df = pd.read_csv(path)
            model_results[model_name] = df
            print(f"  ✓ {model_name:<12} 读取: {path}")
        else:
            print(f"  ✗ {model_name:<12} 找不到: {path}")

    if not model_results:
        continue

    # 以第一个模型的experiment列表为基准
    base_model = list(model_results.keys())[0]
    experiments = model_results[base_model]['experiment'].tolist()

    rows = []
    for exp in experiments:
        row = {
            'Experiment':     exp,
            'Feature Groups': EXPERIMENT_FEATURES.get(exp, exp),
        }
        for model_name, df in model_results.items():
            match = df[df['experiment'] == exp]
            if len(match) > 0:
                row[f'{model_name}_PR-AUC']  = round(match['pr_auc'].values[0], 4)
                row[f'{model_name}_ROC-AUC'] = round(match['roc_auc'].values[0], 4)
            else:
                row[f'{model_name}_PR-AUC']  = None
                row[f'{model_name}_ROC-AUC'] = None
        rows.append(row)

    comparison_df = pd.DataFrame(rows)

    # 标注每行最优模型
    pr_cols = [f'{m}_PR-AUC' for m in model_results.keys()]
    comparison_df['Best Model'] = comparison_df[pr_cols].idxmax(axis=1).str.replace('_PR-AUC', '')
    comparison_df['Best PR-AUC'] = comparison_df[pr_cols].max(axis=1)

    print(f"\n  PR-AUC 对比表:")
    print(comparison_df[['Experiment'] + pr_cols + ['Best Model']].to_string(
        index=False, float_format=lambda x: f'{x:.4f}'))

    path_out = f'results_summary/model_comparison_{target}.csv'
    comparison_df.to_csv(path_out, index=False)
    print(f"\n  ✓ 保存: {path_out}")

    # ── Table 2: 每个模型的最优实验 ──────────────────────────────────────────
    print(f"\n  各模型最优实验:")
    for model_name, df in model_results.items():
        best_row = df.loc[df['pr_auc'].idxmax()]
        print(f"    {model_name:<12} {best_row['experiment']:<30} "
              f"PR-AUC={best_row['pr_auc']:.4f}  ROC-AUC={best_row['roc_auc']:.4f}")

        overall_rows.append({
            'Target':      target,
            'Model':       model_name,
            'Best Exp':    best_row['experiment'],
            'Features':    EXPERIMENT_FEATURES.get(best_row['experiment'], '-'),
            'N':           int(best_row['n_features']),
            'PR-AUC':      round(best_row['pr_auc'], 4),
            'ROC-AUC':     round(best_row['roc_auc'], 4),
            'F1':          round(best_row['f1'], 4),
            'Recall':      round(best_row['recall'], 4),
        })

    # ── Table 3: 最优模型的SHAP ──────────────────────────────────────────────
    # 优先用LightGBM，其次XGBoost
    for model_name in ['LightGBM', 'XGBoost']:
        if model_name not in model_results:
            continue
        df = model_results[model_name]
        best_exp  = df.loc[df['pr_auc'].idxmax(), 'experiment']
        prefix    = MODELS[model_name]
        shap_path = f'{prefix}_{target}/shap_importance.csv'

        if not os.path.exists(shap_path):
            print(f"\n  ✗ SHAP文件不存在: {shap_path}")
            continue

        shap_df = pd.read_csv(shap_path, header=None,
                              names=['Feature', 'SHAP Importance'])
        shap_df = (shap_df.sort_values('SHAP Importance', ascending=False)
                          .head(20).reset_index(drop=True))
        shap_df.index += 1
        shap_df.insert(0, 'Rank', shap_df.index)

        print(f"\n  Top 20 SHAP ({model_name} - {best_exp}):")
        print(shap_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

        shap_out = f'results_summary/shap_{model_name.lower()}_{target}.csv'
        shap_df.to_csv(shap_out, index=False)
        print(f"  ✓ 保存: {shap_out}")
        break  # 只生成一个最优模型的SHAP

# ── 全局汇总表 ────────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  全局汇总（所有模型 × 所有时间窗口）")
print(f"{'='*70}")

overall_df = pd.DataFrame(overall_rows)
if len(overall_df) > 0:
    print(overall_df.to_string(index=False))
    overall_df.to_csv('results_summary/overall_summary.csv', index=False)
    print(f"\n✓ 保存: results_summary/overall_summary.csv")

print(f"\n所有文件保存在: results_summary/")


  EWS_5min
  ✓ XGBoost      读取: results_xgb_EWS_5min/ablation_results.csv
  ✓ LightGBM     读取: results_lgb_EWS_5min/ablation_results.csv
  ✓ Logistic     读取: results_lr_EWS_5min/ablation_results.csv
  ✓ CNN-GRU      读取: results_cnn_gru_EWS_5min/ablation_results.csv

  PR-AUC 对比表:
            Experiment  XGBoost_PR-AUC  LightGBM_PR-AUC  Logistic_PR-AUC  CNN-GRU_PR-AUC Best Model
       baseline_market          0.5250           0.5309           0.2779          0.1790   LightGBM
        M1_add_cascade          0.5008           0.5413           0.2769          0.1820   LightGBM
        M2_add_network          0.5050           0.5717           0.2839          0.2025   LightGBM
       M3_add_temporal          0.5957           0.6155           0.2551          0.1960   LightGBM
           M4_add_text          0.5825           0.6197           0.2577          0.1848   LightGBM
    M5_add_interaction          0.5651              NaN              NaN             NaN    XGBoost
full_with_coordina